test.wav

      │
      ▼
preprocess_audio()

      │
      ▼
extract_features()

      │
      ▼
scaler.transform()

      │
      ▼
model.predict()

      │
      ▼
"Biological"

Importing Libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import joblib

import numpy as np
import pandas as pd

import librosa
import soundfile as sf

from sklearn.preprocessing import StandardScaler

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Checking if my audio file is not already one of the datapoints

In [9]:
test_audio_path = Path("/content/drive/MyDrive/Test/Tanker_9.wav")
data_dir = Path("/content/drive/MyDrive/Underwater Audio Data/data")

# Get the name of the test file
test_audio_filename = test_audio_path.name

# Get all audio file names in the data directory
# Assuming audio files have common extensions like .wav, .mp3, etc.
# Adjust extensions as needed
data_audio_files = [f.name for f in data_dir.glob("*.wav")]

is_present = test_audio_filename in data_audio_files

if is_present:
    print(f"'{test_audio_filename}' is present in '{data_dir}'.")
else:
    print(f"'{test_audio_filename}' is NOT present in '{data_dir}'.")

'Tanker_9.wav' is NOT present in '/content/drive/MyDrive/Underwater Audio Data/data'.


In [5]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Underwater Audio Data")

MODELS_DIR = PROJECT_ROOT / "models"
PROCESSED_DIR = PROJECT_ROOT / "processed"

MODEL_PATH = MODELS_DIR / "logistic_regression.pkl"
SCALER_PATH = MODELS_DIR / "standard_scaler.pkl"

In [10]:
model = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)

print("Logistic Regression Loaded")
print("StandardScaler Loaded")

Logistic Regression Loaded
StandardScaler Loaded


In [11]:
# =====================================
# Audio Processing Constants
# =====================================

TARGET_SR = 16000

WINDOW_SEC = 3.0
HOP_SEC = 1.5

N_MFCC = 13
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128

In [12]:
def window_audio(y, sr, window_sec=WINDOW_SEC, hop_sec=HOP_SEC):
    window_len = int(window_sec * sr)
    hop_len = int(hop_sec * sr)

    if len(y) <= window_len:
        return [np.pad(y, (0, window_len - len(y)))]

    chunks = []

    for start in range(0, len(y) - window_len + 1, hop_len):
        chunks.append(y[start:start + window_len])

    if (len(y) - window_len) % hop_len != 0:
        chunks.append(y[-window_len:])

    return chunks

preprocess_audio()

In [13]:
def preprocess_audio(audio_path):
    """
    Load and preprocess an audio file.
    Returns a list of processed 3-second chunks.
    """

    # Load audio
    y, sr = librosa.load(audio_path, sr=TARGET_SR, mono=True)

    # Skip extremely short audio
    if len(y) < int(0.2 * TARGET_SR):
        raise ValueError("Audio is too short.")

    # Trim silence
    y, _ = librosa.effects.trim(y, top_db=25)

    # Remove DC offset
    y = y - np.mean(y)

    # Peak normalization
    peak = np.max(np.abs(y))
    if peak > 0:
        y = y / peak

    # Split into windows
    chunks = window_audio(y, sr)

    return chunks, sr

extract_features()

In [14]:
def extract_features(y, sr):

    features = {}

    # MFCC
    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=N_MFCC
    )

    mfcc_mean = np.mean(mfcc, axis=1)

    for i, value in enumerate(mfcc_mean):
        features[f"mfcc_{i+1}"] = value

    # Mel Spectrogram
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )

    mel_db = librosa.power_to_db(mel)

    mel_mean = np.mean(mel_db, axis=1)

    for i, value in enumerate(mel_mean):
        features[f"mel_{i+1}"] = value

    # Chroma
    chroma = librosa.feature.chroma_stft(
        y=y,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH
    )

    chroma_mean = np.mean(chroma, axis=1)

    for i, value in enumerate(chroma_mean):
        features[f"chroma_{i+1}"] = value

    # Spectral Centroid
    centroid = librosa.feature.spectral_centroid(
        y=y,
        sr=sr
    )

    features["spectral_centroid"] = np.mean(centroid)

    # Zero Crossing Rate
    zcr = librosa.feature.zero_crossing_rate(y)

    features["zero_crossing_rate"] = np.mean(zcr)

    return features

predict_audio()

In [15]:
from collections import Counter

def predict_audio(audio_path):

    # Preprocess audio
    chunks, sr = preprocess_audio(audio_path)

    # Extract features from each chunk
    feature_list = []

    for chunk in chunks:
        features = extract_features(chunk, sr)
        feature_list.append(features)

    # Convert to DataFrame
    feature_df = pd.DataFrame(feature_list)

    # IMPORTANT:
    # Ensure the feature columns are in the same order as during training.
    # (We'll improve this in the next step.)
    feature_df = feature_df[scaler.feature_names_in_]

    # Scale features
    X_scaled = scaler.transform(feature_df)

    # Predict each chunk
    predictions = model.predict(X_scaled)

    # Majority vote
    final_prediction = Counter(predictions).most_common(1)[0][0]

    return final_prediction

Test the pipeline

In [27]:
audio_path = "/content/drive/MyDrive/Test/test2.wav"

# --- Replicating logic from predict_audio to get probabilities ---
# Preprocess audio
chunks, sr = preprocess_audio(audio_path)

# Extract features from each chunk
feature_list = []
for chunk in chunks:
    features = extract_features(chunk, sr)
    feature_list.append(features)

# Convert to DataFrame
feature_df = pd.DataFrame(feature_list)

# Ensure the feature columns are in the same order as during training.
# This relies on `scaler.feature_names_in_` which is loaded from the trained scaler.
feature_df = feature_df[scaler.feature_names_in_]

# Scale features
X_scaled = scaler.transform(feature_df)

# Predict each chunk and get probabilities
predictions = model.predict(X_scaled)
probabilities = model.predict_proba(X_scaled)

# Majority vote for the final prediction
from collections import Counter # Ensure Counter is imported if not globally available
final_prediction = Counter(predictions).most_common(1)[0][0]

# Calculate average probabilities across all chunks
avg_probabilities = np.mean(probabilities, axis=0)
class_labels = model.classes_ # Get class labels from the trained model
confidence_levels = dict(zip(class_labels, avg_probabilities))
# --- End of replicated logic ---

print("=" * 40)
print("Predicted Class :", final_prediction)
print("Confidence Levels:")
# Sort confidence levels for consistent output
for class_label, confidence in sorted(confidence_levels.items()):
    print(f"  {class_label}: {confidence:.2%}") # Format as percentage
print("=" * 40)

Predicted Class : Biological
Confidence Levels:
  Ambience: 0.08%
  Biological: 99.79%
  Vessels: 0.14%
